# Graph RAG-Tool Fusion - Hands-on Build

**Build date:** 2026-06-09
**Paper explainer:** [../graph_rag_tool_fusion.md](../graph_rag_tool_fusion.md)
**Compliance:** Synthetic data only. Decision-support, not live trading. No real client data / MNPI.

## The Business Problem

A wealth-management copilot on a trading platform exposes a catalog of analytics & execution tools (risk attribution, rebalancing, tax-loss harvesting, price lookup, account context). Many silently **depend on others**: `compute_portfolio_risk` needs `get_current_positions`, which needs `resolve_account_id`; `place_rebalance_order` needs `get_live_price`, which needs `resolve_ticker`. Naive vector retrieval fetches the headline tool an advisor asks for but **drops its prerequisite tools**, so the agent stalls mid-task. **Goal:** build Graph RAG-Tool Fusion over a synthetic tool knowledge graph so each advisor query retrieves the right tool *and* its full dependency chain.

> Complete every `# TODO` below, then tick the **Validation Checklist** at the bottom. Finishing it means you've reproduced every mechanism in the paper - that's the comprehension check.

In [ ]:
import os
import json
import operator
from typing import TypedDict, Annotated, List, Dict, Any, Optional, Literal, Tuple

import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pydantic import BaseModel, Field

# LangChain messages / prompts / tools
from langchain_core.messages import (
    HumanMessage, AIMessage, SystemMessage, ToolMessage, BaseMessage
)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool

# LangChain model providers + embeddings
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_mistralai import ChatMistralAI

# LangGraph
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Verify required API keys
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not set'
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY not set'
assert os.getenv('LANGSMITH_API_KEY'), 'LANGSMITH_API_KEY not set'

In [ ]:
# LLM instances (mirrors the repo's new-module template) + embeddings
llmOpenAI = ChatOpenAI(model='gpt-5.4-mini', temperature=0.0)
llmAnthropic = ChatAnthropic(model='claude-haiku-4-5', temperature=0.0)

# Embedding model for first-pass vector retrieval.
# Paper used Azure OpenAI text-embedding-ada-002; we use a current default.
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## Synthetic Tool Catalog (Trading / Wealth-Management)

This is the boilerplate the build runs on - **provided in full** so you can focus on the paper's *mechanisms*, not data wrangling. It encodes 12 tools (3 core, 9 regular) and **all four edge types** from the paper:

| Edge type | Code value | Example in this catalog |
|---|---|---|
| Tool directly depends on | `tool_direct` | `compute_portfolio_risk` -> `get_current_positions` |
| Tool indirectly depends on | `tool_indirect` | `get_market_news` -> `resolve_ticker` |
| Parameter directly depends on | `param_direct` | `get_live_price` needs `ticker` from `resolve_ticker` |
| Parameter indirectly depends on | `param_indirect` | `performance_report` needs `get_current_datetime` only if `period` is relative (e.g. "YTD") |

In [ ]:
# Each tool: name, type ('core'|'regular'), description, args, depends_on edges.
# Edge: {target, edge_type, reason, parameter_name (or None)}.
TOOL_CATALOG = [
    {"name": "resolve_account_id", "type": "core",
     "description": "Resolve a client's internal account id from their name.",
     "args": ["client_name"], "depends_on": []},
    {"name": "resolve_ticker", "type": "core",
     "description": "Resolve a stock ticker symbol from a company name.",
     "args": ["company_name"], "depends_on": []},
    {"name": "get_current_datetime", "type": "core",
     "description": "Return the current date and time.",
     "args": [], "depends_on": []},

    {"name": "get_live_price", "type": "regular",
     "description": "Get the live market price for a stock ticker.",
     "args": ["ticker"],
     "depends_on": [{"target": "resolve_ticker", "edge_type": "param_direct",
                     "reason": "needs a ticker symbol", "parameter_name": "ticker"}]},
    {"name": "get_current_positions", "type": "regular",
     "description": "Get the current holdings for a client account.",
     "args": ["account_id"],
     "depends_on": [{"target": "resolve_account_id", "edge_type": "param_direct",
                     "reason": "needs an account id", "parameter_name": "account_id"}]},
    {"name": "compute_portfolio_risk", "type": "regular",
     "description": "Compute portfolio risk metrics (volatility, VaR) for an account.",
     "args": ["account_id"],
     "depends_on": [{"target": "get_current_positions", "edge_type": "tool_direct",
                     "reason": "needs holdings to compute risk", "parameter_name": None},
                    {"target": "resolve_account_id", "edge_type": "param_direct",
                     "reason": "needs an account id", "parameter_name": "account_id"}]},
    {"name": "risk_attribution", "type": "regular",
     "description": "Attribute portfolio risk to factors and individual holdings.",
     "args": ["account_id"],
     "depends_on": [{"target": "compute_portfolio_risk", "edge_type": "tool_direct",
                     "reason": "needs risk metrics to attribute", "parameter_name": None}]},
    {"name": "tax_loss_harvest_suggestions", "type": "regular",
     "description": "Suggest tax-loss harvesting trades for an account.",
     "args": ["account_id"],
     "depends_on": [{"target": "get_current_positions", "edge_type": "tool_direct",
                     "reason": "needs holdings", "parameter_name": None},
                    {"target": "get_live_price", "edge_type": "tool_direct",
                     "reason": "needs current prices to find losses", "parameter_name": None}]},
    {"name": "rebalance_plan", "type": "regular",
     "description": "Produce a rebalancing plan toward a target allocation.",
     "args": ["account_id", "target_allocation"],
     "depends_on": [{"target": "get_current_positions", "edge_type": "tool_direct",
                     "reason": "needs current holdings", "parameter_name": None},
                    {"target": "get_live_price", "edge_type": "tool_direct",
                     "reason": "needs prices to value the plan", "parameter_name": None}]},
    {"name": "place_rebalance_order", "type": "regular",
     "description": "Place orders to execute a rebalancing plan.",
     "args": ["account_id", "plan"],
     "depends_on": [{"target": "rebalance_plan", "edge_type": "tool_direct",
                     "reason": "needs a plan to execute", "parameter_name": None},
                    {"target": "get_live_price", "edge_type": "tool_direct",
                     "reason": "needs prices to size orders", "parameter_name": None}]},
    {"name": "performance_report", "type": "regular",
     "description": "Generate a performance report for an account over a period.",
     "args": ["account_id", "period"],
     "depends_on": [{"target": "get_current_positions", "edge_type": "tool_direct",
                     "reason": "needs holdings", "parameter_name": None},
                    {"target": "get_current_datetime", "edge_type": "param_indirect",
                     "reason": "only if period is relative, e.g. 'YTD'", "parameter_name": "period"}]},
    {"name": "get_market_news", "type": "regular",
     "description": "Get recent market news for a company.",
     "args": ["company_name"],
     "depends_on": [{"target": "resolve_ticker", "edge_type": "tool_indirect",
                     "reason": "benefits from a ticker but works with the name", "parameter_name": None}]},
]

print(f"{len(TOOL_CATALOG)} tools | core: {sum(t['type']=='core' for t in TOOL_CATALOG)} | regular: {sum(t['type']=='regular' for t in TOOL_CATALOG)}")

## Coverage Map - the comprehension contract

Complete every section below; together they cover **every mechanism** in the paper.

| # | Paper aspect | Role in this build | Section |
|---|---|---|---|
| 1 | Tool schema: core vs regular **nodes** | Validate & type the catalog | S1 |
| 2 | Four **dependency edge types** | Encode/validate edges | S1 |
| 3 | **Graph indexing** | Build the KG adjacency map | S2 |
| 4 | **Embeddings + vector index** | Embed tool descriptions | S3 |
| 5 | **Vector search (top-k)** | First-pass semantic retrieval | S4 |
| 6 | (Optional) **Query transform** | Rewrite query before embed | S5 |
| 7 | (Optional) **Reranking** | LLM puts best tool at #1 | S6 |
| 8 | **Graph traversal (DFS, d_limit)** | Pull each tool's dependencies | S7 |
| 9 | **Ordered concat + dedup + top-K** (k/d/K, min(1,K/N)) | Assemble final tool list | S8 |
| 10 | **Equip to agent + execute** | Run multi-step query with retrieved tools | S9 |
| 11 | **Evaluation (mAP@K)** vs naive RAG | Quantify the lift | S10 |
| 12 | **LangGraph wire-up** | Assemble pipeline as a StateGraph | S11 |

### S1 - Tool schema: nodes (core vs regular) + 4 edge types

The paper models each tool as a **node** (core or regular) with typed **dependency edges**. Pin that contract down so the rest of the build is type-safe.

In [ ]:
EDGE_TYPES = {"tool_direct", "tool_indirect", "param_direct", "param_indirect"}

# TODO:
# - Define a Pydantic `Dependency` (target, edge_type, reason, parameter_name)
#   and `ToolNode` (name, type, description, args, depends_on: list[Dependency]).
# - Parse TOOL_CATALOG into ToolNode objects -> TOOLS: dict[str, ToolNode].
# - Validate: every dependency.target exists in the catalog; every edge_type in EDGE_TYPES.

# class Dependency(BaseModel):
#     ...
# class ToolNode(BaseModel):
#     ...
# TOOLS: dict[str, ToolNode] = {...}


### S2 - Graph indexing

Build the knowledge graph once. DFS will traverse this in S7. The paper uses Neo4j; at this scale a plain adjacency map is enough (swap in `networkx`/Neo4j later if you like).

In [ ]:
# TODO: build adjacency: KG[tool_name] -> list[(target_name, edge_type)]
def build_kg(catalog: list[dict]) -> dict[str, list[tuple[str, str]]]:
    ...  # iterate tools, read depends_on, emit (target, edge_type) edges

# KG = build_kg(TOOL_CATALOG)
# print({k: v for k, v in list(KG.items())[:3]})


### S3 - Embeddings + vector index

Embed each tool's `name + description` (the *semantic* surface). Note: dependencies are deliberately NOT part of this - that's the whole point, they're retrieved structurally in S7, not semantically here.

In [ ]:
# TODO: embed tool name+description; keep parallel arrays.
# TOOL_NAMES: list[str]
# TOOL_VECTORS: np.ndarray  # shape [num_tools, dim], L2-normalized for cosine

# texts = [f"{t['name']}: {t['description']}" for t in TOOL_CATALOG]
# vecs = embeddings.embed_documents(texts)
# TOOL_NAMES = [t['name'] for t in TOOL_CATALOG]
# TOOL_VECTORS = ...  # np.array(vecs), then normalize


### S4 - Vector search (top-k)

First pass: rank tools by cosine similarity to the query, return the top-k names. This is exactly what *naive RAG* does - and where it stops.

In [ ]:
# TODO:
def vector_search(query: str, top_k: int = 3) -> list[str]:
    # embed query (embeddings.embed_query), cosine vs TOOL_VECTORS, return top_k TOOL_NAMES
    ...

# vector_search("What is the risk on Jane Doe's portfolio?", top_k=3)


### S5 - (Optional) Query transformation

Advanced-RAG hook: rewrite/expand the advisor query before embedding. Return the query unchanged to disable. The paper notes any pre-retrieval trick can stack here.

In [ ]:
# TODO (optional):
def query_transform(query: str) -> str:
    # e.g. prompt llmOpenAI to rewrite into a tool-search-friendly query; else return query
    return query


### S6 - (Optional) Reranking

LLM reranks the initial vector hits so the **most relevant tool is #1**. In the paper this adds +7-14% mAP@10 and cuts truncation errors by 52%, because the #1 tool's dependency chain is expanded first and survives the top-K cut.

In [ ]:
class RerankResult(BaseModel):
    ordered_tools: list[str] = Field(description="candidate tool names, most relevant first")

# TODO (optional):
def rerank(query: str, candidates: list[str], rerank_top_k: int = 3) -> list[str]:
    # use llmOpenAI.with_structured_output(RerankResult) to reorder `candidates`
    ...


### S7 - Graph traversal (DFS, d_limit)

The core contribution. For each vector-retrieved tool, walk its dependency edges (depth-first) to gather prerequisites - even those **semantically unrelated** to the query. Cap at `d_limit` dependencies per starting tool.

In [ ]:
# TODO:
def dfs_dependencies(tool: str, kg: dict, d_limit: int = 10) -> list[str]:
    # DFS from `tool` over kg edges; return dependency names in traversal order,
    # no duplicates, no more than d_limit. (Watch for cycles.)
    ...

# dfs_dependencies("compute_portfolio_risk", KG)
# expect: ['get_current_positions', 'resolve_account_id', ...]


### S8 - Ordered concat + dedup + top-K truncation

Assemble the final list in the paper's order: **top vector tool, then its dependencies, then the next vector tool and its dependencies**, dedup preserving order, truncate to `final_top_K`.

> The accuracy model is `E[GRTF] = E[vector(k)] + E[KG_dep(k,d)] x min(1, K/N)`. The `min(1, K/N)` term is the **truncation risk**: if discovered tools `N` exceed `K`, correct dependencies get dropped. Keep `K` generous for deep chains.

In [ ]:
# TODO: the full Graph RAG-Tool Fusion retrieval (Algorithm 1).
def graph_rag_tool_fusion(query: str, top_k: int = 3, d_limit: int = 10,
                          final_top_K: int = 10, use_query_transform: bool = False,
                          use_rerank: bool = False) -> list[str]:
    # 1) optional query_transform
    # 2) hits = vector_search(top_k)
    # 3) optional rerank(hits)
    # 4) for each hit: append hit, then dfs_dependencies(hit)
    # 5) dedup preserving order
    # 6) truncate to final_top_K
    ...

# graph_rag_tool_fusion("Rebalance Jane Doe's account to 60/40", use_rerank=True)


### S9 - Equip the retrieved tools to an agent + execute

Bind ONLY the GRTF-retrieved subset to the LLM and answer a multi-step advisor query. Implement each catalog tool as a synthetic `@tool` (canned/derived returns - no real market access).

In [ ]:
# TODO:
# - Implement catalog tools as synthetic @tool functions (e.g. resolve_account_id -> 'ACC-1001').
# - retrieved = graph_rag_tool_fusion(user_query, use_rerank=True)
# - bind only those tools: llmOpenAI.bind_tools([tool_fns[name] for name in retrieved])
# - run the manual tool loop (extract tool_calls -> invoke -> ToolMessage -> re-invoke)
#   OR use langchain `create_agent`. Confirm the agent never stalls on a missing dependency.


### S10 - Evaluation: mAP@K, GRTF vs naive RAG

Define golden tool sets (main tool **plus all dependencies**) per query, then compare naive RAG (vector-only) against GRTF. You should reproduce the paper's headline: GRTF >> naive RAG, and the gap widens with dependency depth.

In [ ]:
# Synthetic eval set: query -> golden tools (main + every dependency it needs).
EVAL = [
    {"query": "What's the risk on Jane Doe's portfolio?",
     "golden": ["compute_portfolio_risk", "get_current_positions", "resolve_account_id"]},
    {"query": "Suggest tax-loss harvesting trades for client Acme Trust.",
     "golden": ["tax_loss_harvest_suggestions", "get_current_positions",
                "resolve_account_id", "get_live_price", "resolve_ticker"]},
    {"query": "Place a rebalance order for Jane Doe to 60/40.",
     "golden": ["place_rebalance_order", "rebalance_plan", "get_current_positions",
                "resolve_account_id", "get_live_price", "resolve_ticker"]},
]

def average_precision(retrieved: list[str], golden: list[str], k: int = 10) -> float:
    ...  # TODO: standard AP@k

# TODO: compute mAP@10 for:
#   naive_rag(q)  = vector_search(q, top_k=10)
#   grtf(q)       = graph_rag_tool_fusion(q, top_k=3, final_top_K=10, use_rerank=True)
# Print both; expect GRTF clearly higher.


### S11 - Wire it into a LangGraph StateGraph

Assemble the pipeline as an explicit graph so each stage is a node you can trace in LangSmith.

In [ ]:
class GRTFState(TypedDict):
    query: str
    transformed: str
    vector_hits: list[str]
    reranked: list[str]
    retrieved_tools: list[str]
    answer: str

# TODO: build nodes -> transform -> vector_search -> rerank -> traverse_truncate -> agent_execute
# g = StateGraph(GRTFState)
# g.add_node(...) ; g.add_edge(START, ...) ; ... ; g.add_edge(..., END)
# app = g.compile()
# app.invoke({"query": "Rebalance Jane Doe's account to 60/40"})


## Validation Checklist

Tick each box only when that mechanism runs end-to-end. A full tick = you've reproduced the paper.

- [ ] **S1** Catalog parses into typed nodes; all 4 edge types validated; bad targets rejected
- [ ] **S2** KG adjacency built; spot-checked a few tools' edges
- [ ] **S3** Tool vectors built and normalized
- [ ] **S4** `vector_search` returns sensible top-k (this is the naive-RAG baseline)
- [ ] **S5** `query_transform` runs (or is a deliberate pass-through)
- [ ] **S6** `rerank` reorders candidates so the best tool is #1
- [ ] **S7** `dfs_dependencies` returns prerequisites - including semantically unrelated ones (e.g. `resolve_account_id`)
- [ ] **S8** `graph_rag_tool_fusion` returns an ordered, deduped, top-K list; truncation behaves as `min(1, K/N)`
- [ ] **S9** Agent answers a multi-step query using only retrieved tools and never stalls on a missing dependency
- [ ] **S10** mAP@10 computed; **GRTF clearly beats naive RAG**, gap grows with dependency depth
- [ ] **S11** Full pipeline runs as a LangGraph `StateGraph` and traces in LangSmith

**Reflection (write 2-3 sentences):** Where did naive RAG fail that GRTF fixed, and which knob (`k`, `d`, `K`, rerank) moved your mAP the most?